In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from jcopml.pipeline import num_pipe, cat_pipe
from jcopml.utils import save_model, load_model
from jcopml.plot import plot_missing_value
from jcopml.feature_importance import mean_score_decrease

In [2]:
df = pd.read_csv("data/Pgr_Characters.csv")
df.head()

,Character,Rarity,Element,Class,Signature
0,Lucia: Lotus,B-Rank,Physical,Attacker,Lotus Berserker
1,Liv: Eclipse,B-Rank,Physical,Support,Type Zero
2,Nanami: Storm,B-Rank,Physical,Tank,Inverse - Chimera
3,Lucia: Dawn,A-Rank,Lightning,Attacker,Inverse - Shadow
4,Liv: Lux,A-Rank,Lightning,Support,Benediction


In [3]:
# 1. Cek jumlah data per kelas
print(df["Class"].value_counts())

# 2. Buang baris yang kelasnya hanya punya kurang dari 2 anggota
class_counts = df["Class"].value_counts()
valid_classes = class_counts[class_counts >= 2].index
df = df[df["Class"].isin(valid_classes)].reset_index(drop=True)

# Lanjutkan dengan kode X dan y kamu seperti biasa...
X = df.drop(columns=["Character", "Class"])
y = df["Class"]

Class
Attacker       23
Tank           18
Support         9
Vanguard        7
Amplifier       5
Annihilator     1
Name: count, dtype: int64


In [4]:
X = df.drop(columns=[
    "Character",
    "Rarity"
])
y = df["Rarity"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((49, 3), (13, 3), (49,), (13,))

In [5]:
preprocessor = ColumnTransformer([
    ('numeric', num_pipe(), []),
    ('categoric', cat_pipe(encoder='onehot'), ["Class", "Element", "Signature"]),
])

In [6]:
from sklearn.linear_model import LogisticRegression
pipeline = Pipeline([
    ('prep', preprocessor),
    ('algo', LogisticRegression(solver='lbfgs', n_jobs=-1, random_state=42 ,C=0.5, class_weight="balanced"))
])

In [7]:
from sklearn.model_selection import LeaveOneOut, RandomizedSearchCV
from jcopml.tuning import random_search_params as rsp

model = RandomizedSearchCV(pipeline, rsp.logreg_params, cv=2, scoring='accuracy', n_iter=25, n_jobs=-1, verbose=1, random_state=42)
model.fit(X_train, y_train)

print(model.best_params_)
print(model.score(X_train, y_train), model.best_score_, model.score(X_test, y_test))

Fitting 2 folds for each of 25 candidates, totalling 50 fits
{'algo__C': np.float64(4.676478725076048), 'algo__fit_intercept': True}
1.0 0.5508333333333333 0.5384615384615384


In [8]:
save_model(model, "pgr_over.pkl")

Model is pickled as model/pgr_over.pkl
